# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

## Solución

### 1. Explorar la estructura del HTML

Antes de escribir el scraper, inspeccionamos una página del catálogo y la ficha de un libro.

In [1]:
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/"

response = requests.get(BASE_URL + "page-1.html", timeout=20)
print(response.status_code)
print("Codificación que adivina requests:", response.encoding)
print("Codificación real:", response.apparent_encoding)

200
Codificación que adivina requests: ISO-8859-1
Codificación real: utf-8


⚠️ La web no declara bien su codificación: `requests` cree que es `ISO-8859-1` y el símbolo `£` saldría como `Â£`. Para evitarlo, a BeautifulSoup le pasamos los **bytes** (`response.content`) y no el texto (`response.text`). Así detecta la codificación correcta por sí mismo.

In [2]:
soup = BeautifulSoup(response.content, "html.parser")
first_book = soup.select_one("article.product_pod")
print(first_book.prettify()[:1200])

<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



En la página del catálogo cada libro es un `<article class="product_pod">` con:
- **Rating** → la clase de `<p class="star-rating Three">` (en palabras, hay que pasarlo a número)
- **Título completo** → el atributo `title` del enlace (el texto visible está cortado: *"A Light in the ..."*)
- **Precio** → `<p class="price_color">`
- **Enlace a la ficha** → `href` del `<a>`

El **UPC**, el **género**, la **disponibilidad** y la **descripción** solo están en la ficha de cada libro:

In [3]:
detail = requests.get(BASE_URL + first_book.h3.a["href"], timeout=20)
detail_soup = BeautifulSoup(detail.content, "html.parser")

print("Breadcrumb:", [li.get_text(strip=True) for li in detail_soup.select("ul.breadcrumb li")])
print({tr.th.text: tr.td.text for tr in detail_soup.select("table.table-striped tr")})
print("Descripción:", detail_soup.select_one("#product_description ~ p").text[:100], "...")

Breadcrumb: ['Home', 'Books', 'Poetry', 'A Light in the Attic']
{'UPC': 'a897fe39b1053632', 'Product Type': 'Books', 'Price (excl. tax)': '£51.77', 'Price (incl. tax)': '£51.77', 'Tax': '£0.00', 'Availability': 'In stock (22 available)', 'Number of reviews': '0'}
Descripción: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and ...


El género es el **tercer elemento del breadcrumb** (`Home > Books > Poetry > título`) y el UPC y la disponibilidad están en la tabla de información del producto.

### 2. La función `scrape_books`

**Estrategia:** el rating y el precio ya aparecen en el catálogo, así que **filtramos ahí** y solo entramos en la ficha de los libros que cumplen los criterios. Así hacemos ~50 peticiones de catálogo + unas decenas de fichas, en vez de abrir los 1000 libros uno a uno. Es más rápido y respetuoso con el servidor.

Otras buenas prácticas que aplico:
- `requests.Session()` reutiliza la conexión entre peticiones.
- `timeout` y `raise_for_status()`: si la web falla, el error salta en vez de quedarse colgado o devolver datos vacíos.
- Una pequeña pausa entre peticiones para no saturar el servidor.
- Sigo el enlace **"next"** en vez de dar por hecho que hay 50 páginas: si la web crece, el código sigue funcionando.

In [4]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def get_soup(session, url, delay=0.1):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    time.sleep(delay)
    return BeautifulSoup(response.content, "html.parser")


def parse_catalog_page(soup):
    """Extrae título, precio, rating y enlace de cada libro de una página del catálogo."""
    books = []
    for article in soup.select("article.product_pod"):
        rating_word = [c for c in article.select_one("p.star-rating")["class"] if c != "star-rating"][0]
        books.append({
            "Title": article.h3.a["title"],
            "Price (£)": float(article.select_one("p.price_color").text.strip().lstrip("£")),
            "Rating": RATING_MAP[rating_word],
            "url": BASE_URL + article.h3.a["href"],
        })
    return books


def parse_book_page(soup):
    """Extrae UPC, género, disponibilidad y descripción de la ficha de un libro."""
    info = {tr.th.text: tr.td.text for tr in soup.select("table.table-striped tr")}
    description = soup.select_one("#product_description ~ p")
    return {
        "UPC": info["UPC"],
        "Genre": soup.select("ul.breadcrumb li")[2].get_text(strip=True),
        "Availability": info["Availability"],
        "Description": description.get_text(strip=True) if description else None,
    }


def scrape_books(min_rating, max_price):
    """Devuelve un DataFrame con los libros de Books to Scrape con rating >= min_rating y precio <= max_price."""
    session = requests.Session()

    # 1) Recorremos el catálogo siguiendo el botón "next" y filtramos por rating y precio
    candidates = []
    url = BASE_URL + "page-1.html"
    while url:
        soup = get_soup(session, url)
        candidates += [
            b for b in parse_catalog_page(soup)
            if b["Rating"] >= min_rating and b["Price (£)"] <= max_price
        ]
        next_link = soup.select_one("li.next a")
        url = BASE_URL + next_link["href"] if next_link else None

    # 2) Solo para los libros que pasan el filtro, entramos en su ficha
    books = [{**book, **parse_book_page(get_soup(session, book["url"]))} for book in candidates]

    columns = ["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"]
    return pd.DataFrame(books, columns=columns)

### 3. Ejecutamos con rating ≥ 4 y precio ≤ £20

In [5]:
%%time
books = scrape_books(min_rating=4, max_price=20)
print(books.shape)
books.head()

(75, 7)
CPU times: total: 2.86 s
Wall time: 31.1 s


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."


### 4. Comprobamos que los datos son correctos

In [6]:
print(books.dtypes, "\n")
print("Nulos por columna:\n", books.isnull().sum(), "\n")
print("UPC repetidos:", books["UPC"].duplicated().sum())
print("Rating mínimo:", books["Rating"].min(), "| Precio máximo:", books["Price (£)"].max())

UPC                 str
Title               str
Price (£)       float64
Rating            int64
Genre               str
Availability        str
Description         str
dtype: object 

Nulos por columna:
 UPC             0
Title           0
Price (£)       0
Rating          0
Genre           0
Availability    0
Description     0
dtype: int64 

UPC repetidos: 0
Rating mínimo: 4 | Precio máximo: 19.69


In [7]:
# Libros sin descripción en la web (el scraper devuelve None en vez de fallar)
books[books["Description"].isnull()]

,UPC,Title,Price (£),Rating,Genre,Availability,Description


### 5. Probamos otros parámetros

El enunciado pide comprobar que la función funciona con otras búsquedas. Probamos un caso más estricto: solo libros de 5 estrellas por menos de £15.

In [8]:
top_cheap = scrape_books(min_rating=5, max_price=15)
print(top_cheap.shape)
assert (top_cheap["Rating"] == 5).all() and (top_cheap["Price (£)"] <= 15).all()
top_cheap[["Title", "Price (£)", "Rating", "Genre"]]

(23, 7)


,Title,Price (£),Rating,Genre
0,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art
1,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy
2,The Third Wave: An Entrepreneur’s Vision of th...,12.61,5,Business
3,Agnostic: A Spirited Manifesto,12.51,5,Nonfiction
4,Walt Disney's Alice in Wonderland,12.96,5,Childrens
5,Superman Vol. 1: Before Truth (Superman by Gen...,11.89,5,Sequential Art
6,Old School (Diary of a Wimpy Kid #10),11.83,5,Humor
7,"Hyperbole and a Half: Unfortunate Situations, ...",14.75,5,Humor
8,Greek Mythic History,10.23,5,Default
9,Dear Mr. Knightley,11.21,5,Fiction


### 6. Análisis rápido de los resultados

La disponibilidad viene como texto (`"In stock (22 available)"`). Sacamos el número de unidades a una columna numérica para poder analizarla.

In [9]:
books["Stock"] = books["Availability"].str.extract(r"\((\d+) available\)").astype(int)
books[["Availability", "Stock"]].head()

,Availability,Stock
0,In stock (19 available),19
1,In stock (18 available),18
2,In stock (18 available),18
3,In stock (16 available),16
4,In stock (16 available),16


In [10]:
genre_summary = books.groupby("Genre").agg(
    books=("UPC", "count"),
    avg_price=("Price (£)", "mean"),
    avg_rating=("Rating", "mean"),
    total_stock=("Stock", "sum"),
).round(2).sort_values("books", ascending=False)
genre_summary

,books,avg_price,avg_rating,total_stock
Genre,,,,
Default,10,14.88,4.70,77
Sequential Art,9,15.24,4.56,104
Young Adult,7,13.74,4.71,51
Nonfiction,6,14.70,4.50,58
Fiction,5,13.16,4.60,59
Poetry,4,15.34,4.50,42
Food and Drink,4,12.20,4.50,37
Fantasy,3,14.24,4.67,28
Romance,3,15.65,4.33,42


In [11]:
books["Rating"].value_counts().sort_index()

Rating
4    33
5    42
Name: count, dtype: int64

In [12]:
books.to_csv("books_rating4_max20.csv", index=False)